# Support Vector Machines (SVMs) for Grain Disease Recognition

Welcome to the next phase of our agricultural analysis. We have gathered a massive dataset of rice and maize images. These staple crops are vital for food security. Our goal is to build an intelligent system that can look at a photograph of a grain and instantly diagnose its health. 

Previously we used Logistic Regression to draw simple lines between healthy and infected crops. Today we are upgrading our mathematical lens. We are deploying Support Vector Machines to find complex boundaries in our data.

## Data Preprocessing

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import os
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    classification_report,
)
from tqdm import tqdm
import joblib
import pandas as pd

from logistic_regression_lib import collect_embeddings

Before we can train our model we must define the world it will operate in. We have two main crops and several categories of health and disease for each. We will define these lists so our system knows exactly what to look for.

In [ ]:
GRAIN_TYPES = ["maize", "rice"]
MAIZE_CATEGORIES = ["0_NOR", "1_F&S", "2_SD", "3_MY", "4_AP", "5_BN", "6_HD"]
RICE_CATEGORIES = ["0_NOR", "1_F&S", "2_SD", "3_MY", "4_AP", "5_BN", "6_UN"]

A machine cannot look at a photograph the way a human does. We need to translate the visual world into numbers. 

By using our feature extraction script we analyze the HSV color space to understand the exact hues of the grain surface. We also compute HOG features to capture the physical texture and structural damage of the crops. This transforms every photograph into a dense 512 number feature vector. Let us collect this data for our training phase.

In [ ]:
# Collect training data
X_train_maize, y_train_maize = collect_embeddings("maize", "train")
X_train_rice, y_train_rice = collect_embeddings("rice", "train")
X_val_maize, y_val_maize = collect_embeddings("maize", "val")
X_val_rice, y_val_rice = collect_embeddings("rice", "val")

# Combine everything into our main training set
X_train = np.concatenate((X_train_maize, X_train_rice, X_val_maize, X_val_rice))
y_train = np.concatenate((y_train_maize, y_train_rice, y_val_maize, y_val_rice))

print(f"\nTraining data shape: {X_train.shape}")
print(f"Number of classes: {len(set(y_train))}")

We cannot evaluate our model using the same images it studied during training. We must gather a completely separate set of photographs to serve as our final test.

In [ ]:
X_test_maize, y_test_maize = collect_embeddings("maize", "test")
X_test_rice, y_test_rice = collect_embeddings("rice", "test")
X_test = np.concatenate((X_test_maize, X_test_rice))
y_test = np.concatenate((y_test_maize, y_test_rice))

print(f"Test data shape: {X_test.shape}")

To make the math work seamlessly we need to convert our text labels into numerical codes. The Label Encoder handles this translation perfectly.

In [ ]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

## Constructing the SVM Pipeline
Here is where the real magic happens. Support Vector Machines work by projecting our data into higher dimensions and finding the optimal hyperplane that separates the different diseases.

Before we can train our model we need to build the assembly line. This pipeline ensures every image feature vector is perfectly scaled before the algorithm even sees it. We also define the exact limits of our experiment by setting up a grid of possible settings for the machine to explore.

In [ ]:
# Set up the scaling and model pipeline
pipeline = Pipeline(
    [("scaler", StandardScaler()), ("classifier", SVC(random_state=42))]
)

# The parameter grid for our search
param_dist = {
    "classifier__C": [0.1, 1, 10, 100],
    "classifier__kernel": ["linear", "rbf"],
    "classifier__gamma": ["scale", "auto", 0.1, 0.01],
    "classifier__class_weight": [None, "balanced"],
}

## Tuning Hyperparameters
We then take the pipeline and the parameters we just built and feed them into the randomized search. Instead of testing every single combination we use a randomized search to save computation time. The algorithm will blindly test 20 different configurations to learn which combination of mathematical boundaries best separates the healthy maize and rice from the diseased ones.

In [ ]:
n_iter = 20

print("Starting Random Search for SVM...")

# Configure the search parameters
random_search = RandomizedSearchCV(
    pipeline,
    param_dist,
    n_iter=n_iter,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=2,
    random_state=42,
)

# Train the model and find the best settings
random_search.fit(X_train, y_train_encoded)

print("\nRANDOM SEARCH RESULTS")
print(f"\nBest hyperparameters: {random_search.best_params_}")
print(f"Best cross-validation F1-score: {random_search.best_score_:.4f}")

## Evaluation of Model
The machine has finished its training phase. Now we pull the absolute best model from the search and pit it against our unseen test photographs. We calculate the precision and recall to generate a comprehensive report card so we can see if our new algorithm outperformed our previous logistic regression efforts. Finally, we securely store the trained model for future use.

In [ ]:
best_pipeline = random_search.best_estimator_

y_test_pred_best = best_pipeline.predict(X_test)

best_accuracy = best_pipeline.score(X_test, y_test_encoded)
best_f1 = f1_score(y_test_encoded, y_test_pred_best, average="weighted")
best_recall = recall_score(y_test_encoded, y_test_pred_best, average="weighted")
best_precision = precision_score(y_test_encoded, y_test_pred_best, average="weighted")

print("\nTEST SET PERFORMANCE")
print(f"Test Accuracy:  {best_accuracy:.4f}")
print(f"Test F1-Score:  {best_f1:.4f}")
print(f"Test Recall:    {best_recall:.4f}")
print(f"Test Precision: {best_precision:.4f}")

print("\nPER-CLASS METRICS")
best_report = classification_report(
    y_test_encoded, y_test_pred_best, target_names=label_encoder.classes_, digits=4
)
print(best_report)

# Securely saving the model
joblib.dump(best_pipeline, "models/svm_tuned.pkl")
print("\nBest tuned pipeline saved to models/svm_tuned.pkl")